# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatimahmahmood/flyrank_ml_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### My rule

I will prioritize content that has not been updated for 91–180 days and still receives meaningful search impressions. The score combines staleness and search volume so that content with stronger visibility receives higher priority.

The 181+ day group was not given the highest staleness score because its overall declining rate was lower than the 91–180 day group. The very-high rates in the 181+ high-volume cells were based on very small sample sizes, so I did not use them as strong evidence.

### Score

- 0–30 days stale: 0 points
- 31–90 days stale: 0 points
- 91–180 days stale: 2 points
- 181+ days stale: 1 point

Volume points:

- Low: 0 points
- Medium: 1 point
- High: 2 points
- Very High: 2 points

Total score = staleness points + volume points.

### Reason codes

- `stale_high_volume` — content is 91+ days old and has high or very-high search volume.
- `stale_medium_volume` — content is 91+ days old and has medium search volume.
- `high_volume_not_stale` — content has high search volume but is not yet in the main stale range.
- `other` — does not match the higher-priority conditions.

### Actions

- Score 4 → REFRESH
- Score 2–3 → REVIEW
- Score 0–1 → MONITOR

In [44]:
from google.colab import userdata

import os

HF_TOKEN = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = HF_TOKEN

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [45]:
import duckdb

con = duckdb.connect()

print("DuckDB connected")

DuckDB connected


In [46]:
import os

con.execute("""INSTALL httpfs; LOAD httpfs; """)

con.execute("""CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN ?) """, [os.environ["HF_TOKEN"]])

print("Hugging Face access configured")

Hugging Face access configured


In [29]:
result = con.sql("""SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""")

result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────────┬────────────┐
│  rows   │ first_date │ last_date  │
│  int64  │    date    │    date    │
├─────────┼────────────┼────────────┤
│ 9841378 │ 2026-03-01 │ 2026-03-31 │
└─────────┴────────────┴────────────┘

In [30]:
con.sql("""
DESCRIBE
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
)
""")

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

In [31]:
march_content = con.sql("""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position
            END
        ) AS avg_position
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_impressions,
    m.march_clicks,
    m.avg_position,
    c.content_created_date,
    c.content_updated_date,
    c.content_type,
    c.is_published,
    c.is_deleted,
    DATE '2026-03-31' - c.content_updated_date AS days_since_update
FROM march m
LEFT JOIN read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
) c
    ON m.client_hash_id = c.client_hash_id
    AND m.content_hash_id = c.content_hash_id
WHERE c.is_deleted = FALSE
  AND c.is_published = TRUE
""")

df = march_content.df()

print("Rows:", len(df))
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 321106


,client_hash_id,content_hash_id,march_impressions,march_clicks,avg_position,content_created_date,content_updated_date,content_type,is_published,is_deleted,days_since_update
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,4.888929,2026-02-12,2026-06-29,keyword article,True,False,-90
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,602.0,4.0,4.428747,2026-02-12,2026-06-29,keyword article,True,False,-90
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,810.0,1.0,4.866123,2026-02-12,2026-06-29,keyword article,True,False,-90
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,82.0,0.0,10.100347,2026-02-12,2026-06-29,keyword article,True,False,-90
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,1858.0,6.0,1.854929,2026-02-12,2026-06-29,keyword article,True,False,-90


In [32]:
import pandas as pd

df = pd.read_csv(
    "/content/flyrank_ml_internship/data/raw/content_refresh_anonymized.csv"
)

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [33]:
!git clone https://github.com/fatimahmahmood/flyrank_ml_internship.git

fatal: destination path 'flyrank_ml_internship' already exists and is not an empty directory.


In [34]:
import pandas as pd

df = pd.read_csv(path)

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [35]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [36]:
df[
    ["days_since_last_update", "impressions_90d"]
].describe()

,days_since_last_update,impressions_90d
count,30000.000000,30000.000000
mean,46.098300,5200.366300
std,42.078709,16838.019547
min,1.000000,1.000000
25%,20.000000,81.000000
50%,20.000000,731.000000
75%,104.000000,3615.250000
max,373.000000,517715.000000


In [37]:
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

staleness_check = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          declining_rate=("trend_direction", lambda x: (x == "down").mean())
      )
      .reset_index()
)

staleness_check

,staleness_bucket,n,declining_rate
0,0-30 days,20480,0.511377
1,31-90 days,175,0.588571
2,91-180 days,9171,0.611057
3,181+ days,174,0.471264


In [38]:
df["volume_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"],
    duplicates="drop"
)

volume_check = (
    df.groupby("volume_bucket", observed=False)
      .agg(
          n=("content_id", "size"),
          declining_rate=("trend_direction", lambda x: (x == "down").mean())
      )
      .reset_index()
)

volume_check

,volume_bucket,n,declining_rate
0,Low,7503,0.376116
1,Medium,7499,0.604614
2,High,7498,0.625634
3,Very High,7500,0.562000


In [39]:
combined_check = (
    df.groupby(["staleness_bucket", "volume_bucket"], observed=False)
      .agg(
          n=("content_id", "size"),
          declining_rate=("trend_direction", lambda x: (x == "down").mean())
      )
      .reset_index()
)

combined_check

,staleness_bucket,volume_bucket,n,declining_rate
0,0-30 days,Low,6350,0.354016
1,0-30 days,Medium,5188,0.585197
2,0-30 days,High,4671,0.600942
3,0-30 days,Very High,4271,0.557715
4,31-90 days,Low,15,0.400000
5,31-90 days,Medium,92,0.695652
6,31-90 days,High,45,0.555556
7,31-90 days,Very High,23,0.347826
8,91-180 days,Low,1002,0.512974
9,91-180 days,Medium,2196,0.646630


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*
### Baseline scoring

I calculate a transparent score from staleness and 90-day impressions. The score uses only pre-outcome information and does not use trend direction or trend percentage. Items are ranked from highest to lowest score.

In [40]:
df["volume_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"],
    duplicates="drop"
)

print("volume_bucket created successfully")
print(df["volume_bucket"].value_counts())

volume_bucket created successfully
volume_bucket
Low          7503
Very High    7500
Medium       7499
High         7498
Name: count, dtype: int64


In [41]:
# Staleness points
df["stale_points"] = 0

df.loc[
    df["days_since_last_update"].between(91, 180),
    "stale_points"
] = 2

df.loc[
    df["days_since_last_update"] >= 181,
    "stale_points"
] = 1


# Volume points
df["volume_points"] = 0

df.loc[
    df["volume_bucket"] == "Medium",
    "volume_points"
] = 1

df.loc[
    df["volume_bucket"].isin(["High", "Very High"]),
    "volume_points"
] = 2


# Final score
df["score"] = df["stale_points"] + df["volume_points"]


# Reason code
df["reason_code"] = "other"

df.loc[
    (df["days_since_last_update"] >= 91) &
    (df["volume_bucket"].isin(["High", "Very High"])),
    "reason_code"
] = "stale_high_volume"

df.loc[
    (df["days_since_last_update"] >= 91) &
    (df["volume_bucket"] == "Medium"),
    "reason_code"
] = "stale_medium_volume"

df.loc[
    (df["days_since_last_update"] < 91) &
    (df["volume_bucket"].isin(["High", "Very High"])),
    "reason_code"
] = "high_volume_not_stale"


# Action
df["action"] = "MONITOR"

df.loc[
    df["score"].between(2, 3),
    "action"
] = "REVIEW"

df.loc[
    df["score"] == 4,
    "action"
] = "REFRESH"


# Rank
queue = df.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = range(1, len(queue) + 1)


# Keep the important columns
output = queue[
    [
        "rank",
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d"
    ]
]

output.head(20)

,rank,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d
0,1,content_5fe46e04994d,client_4e07408562,4,stale_high_volume,REFRESH,104,517715
1,2,content_2dba2b1f9536,client_6208ef0f77,4,stale_high_volume,REFRESH,104,443434
2,3,content_2c2606c5d176,client_19581e27de,4,stale_high_volume,REFRESH,104,347399
3,4,content_cb112fce36be,client_19581e27de,4,stale_high_volume,REFRESH,104,309910
4,5,content_9532f197bbc8,client_4e07408562,4,stale_high_volume,REFRESH,104,309192
5,6,content_36ff89c8214e,client_19581e27de,4,stale_high_volume,REFRESH,104,295097
6,7,content_b28d1efd668f,client_6208ef0f77,4,stale_high_volume,REFRESH,104,286608
7,8,content_813e88069237,client_6208ef0f77,4,stale_high_volume,REFRESH,104,233561
8,9,content_c21024970297,client_19581e27de,4,stale_high_volume,REFRESH,104,211366
9,10,content_c8e9d6ab9013,client_19581e27de,4,stale_high_volume,REFRESH,104,208678


## 3. Top-20 review

I reviewed the 20 highest-ranked items produced by the baseline rule. For each item, I recorded the action, reason, confidence, and what could make the recommendation wrong. The review is intended to identify possible false positives rather than assume that every high-scoring item needs action.

### Top-20 review

1. **REFRESH — stale_high_volume.** High confidence because the content is 104 days since its last update and has 517,715 impressions. It could be wrong if the content is still accurate and does not need updating.

2. **REFRESH — stale_high_volume.** High confidence because the content is 104 days since its last update and has 443,434 impressions. It could be wrong if the content is still accurate despite being old.

3. **REFRESH — stale_high_volume.** High confidence because the content is 104 days since its last update and has 347,399 impressions. It could be wrong if the page does not actually need a content refresh.

4. **REFRESH — stale_high_volume.** High confidence because the content is 104 days old and has 309,910 impressions. It could be wrong if the update date does not reflect the true freshness of the content.

5. **REFRESH — stale_high_volume.** High confidence because the content is 104 days old and has 309,192 impressions. It could be wrong if the content remains useful and accurate.

6. **REFRESH — stale_high_volume.** High confidence because the content is 104 days old and has 295,097 impressions. It could be wrong if high impressions do not mean the page needs refreshing.

7. **REFRESH — stale_high_volume.** High confidence because the content is 104 days old and has 286,608 impressions. It could be wrong if the content is already performing well without an update.

8. **REFRESH — stale_high_volume.** High confidence because the content is 104 days old and has 233,561 impressions. It could be wrong if the page is still accurate and relevant.

9. **REFRESH — stale_high_volume.** High confidence because the content is 104 days old and has 211,366 impressions. It could be wrong if the content does not require changes.

10. **REFRESH — stale_high_volume.** High confidence because the content is 104 days old and has 208,678 impressions. It could be wrong if the page is already meeting its purpose.

11. **REFRESH — stale_high_volume.** High confidence because the content is 104 days old and has 205,915 impressions. It could be wrong if the page is accurate despite its age.

12. **REFRESH — stale_high_volume.** High confidence because the content is 104 days old and has 201,584 impressions. It could be wrong if the content does not need updating.

13. **REFRESH — stale_high_volume.** High confidence because the content is 104 days old and has 201,111 impressions. It could be wrong if the update date is not a good measure of actual content freshness.

14. **REFRESH — stale_high_volume.** High confidence because the content is 104 days old and has 192,205 impressions. It could be wrong if the page remains accurate and useful.

15. **REFRESH — stale_high_volume.** Moderate confidence because the content is 104 days old and has 190,623 impressions. It could be wrong if the high impression count does not indicate a need for refreshing.

16. **REFRESH — stale_high_volume.** Moderate confidence because the content is 104 days old and has 187,893 impressions. It could be wrong if the content is already up to date in substance.

17. **REFRESH — stale_high_volume.** Moderate confidence because the content is 104 days old and has 181,574 impressions. It could be wrong if the page is performing well without needing changes.

18. **REFRESH — stale_high_volume.** Moderate confidence because the content is 104 days old and has 181,514 impressions. It could be wrong if the page is still relevant and accurate.

19. **REFRESH — stale_high_volume.** Moderate confidence because the content is 104 days old and has 179,002 impressions. It could be wrong if the page does not need a refresh despite its age.

20. **REFRESH — stale_high_volume.** Moderate confidence because the content is 104 days old and has 176,296 impressions. It could be wrong if the content is still accurate and useful.

In [42]:
top20 = output.head(20).copy()
top20

,rank,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d
0,1,content_5fe46e04994d,client_4e07408562,4,stale_high_volume,REFRESH,104,517715
1,2,content_2dba2b1f9536,client_6208ef0f77,4,stale_high_volume,REFRESH,104,443434
2,3,content_2c2606c5d176,client_19581e27de,4,stale_high_volume,REFRESH,104,347399
3,4,content_cb112fce36be,client_19581e27de,4,stale_high_volume,REFRESH,104,309910
4,5,content_9532f197bbc8,client_4e07408562,4,stale_high_volume,REFRESH,104,309192
5,6,content_36ff89c8214e,client_19581e27de,4,stale_high_volume,REFRESH,104,295097
6,7,content_b28d1efd668f,client_6208ef0f77,4,stale_high_volume,REFRESH,104,286608
7,8,content_813e88069237,client_6208ef0f77,4,stale_high_volume,REFRESH,104,233561
8,9,content_c21024970297,client_19581e27de,4,stale_high_volume,REFRESH,104,211366
9,10,content_c8e9d6ab9013,client_19581e27de,4,stale_high_volume,REFRESH,104,208678


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

A possible weak pick is Rank 20. It receives a high score because it is 104 days since its last update and has 176,296 impressions. However, age and impressions alone do not prove that the content needs a refresh. The content could still be accurate, relevant, and useful.

The other high-ranked items have the same limitation: the rule identifies pages for review, but it does not prove that a refresh is definitely required.

### Leakage check

The final baseline score uses only `days_since_last_update` and `impressions_90d`.

I did not use `trend_direction` or `trend_pct` to calculate the score. These outcome-related fields were used only during the signal audit.

No future-window outcome was used in the final scoring rule.

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.